# CrisisLens 02: Results

Builds every figure for the README from `outputs/` after `scripts/run_experiments.sh` finishes.
Everything plotted here is **aggregated** (metrics, counts, averages), so the figures are safe to publish.
No tweet text or images are displayed (see `docs/DATA.md`).

Figures are saved to `figures/`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from crisislens.evaluate import load_predictions, softmax
from crisislens.torch_data import label_maps

OUTPUTS = Path("../outputs")          # run from notebooks/
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)
TASKS = ["informative", "humanitarian", "damage"]
KEEP_SEED = 42

# Fixed colors per model (color follows the model, never its rank). Baseline is neutral gray.
COLORS = {"baseline_tfidf_lr": "#8c8b86", "text_bertweet": "#2a78d6", "image_clip": "#eb6834",
          "late_fusion": "#1baf7a", "fusion_concat": "#eda100", "fusion_gated": "#6250d6"}
PRETTY = {"baseline_tfidf_lr": "TF-IDF + LR", "text_bertweet": "BERTweet (text)", "image_clip": "CLIP (image)",
          "late_fusion": "Late fusion", "fusion_concat": "Concat fusion", "fusion_gated": "Gated fusion"}
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "axes.axisbelow": True,
                     "axes.edgecolor": "#8c8b86", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "font.size": 10})

def run_dir(model, seed=KEEP_SEED):
    return OUTPUTS / model / f"seed_{seed}"

## 1. Main results table

In [ ]:
summary = pd.read_csv(OUTPUTS / "results.csv")
table = summary.pivot(index="model", columns="task", values="macro_f1_mean").reindex(
    [m for m in PRETTY if m in set(summary.model)])[[t for t in TASKS if t in set(summary.task)]]
table.rename(index=PRETTY).style.format("{:.3f}").background_gradient(axis=0, cmap="Blues")

In [ ]:
print((OUTPUTS / "results.md").read_text())

## 2. Macro-F1 by model and task (mean ± std over seeds)

In [ ]:
models = [m for m in PRETTY if m in set(summary.model)]
fig, ax = plt.subplots(figsize=(9, 4))
width = 0.8 / len(models)
for i, m in enumerate(models):
    s = summary[summary.model == m].set_index("task").reindex(TASKS)
    x = np.arange(len(TASKS)) + (i - (len(models) - 1) / 2) * width
    ax.bar(x, s.macro_f1_mean, width * 0.9, yerr=s.macro_f1_std, capsize=2, color=COLORS[m],
           label=PRETTY[m], error_kw={"elinewidth": 1, "ecolor": "#52514e"})
ax.set_xticks(range(len(TASKS)), [t.title() for t in TASKS])
ax.set_ylabel("Test macro-F1"); ax.set_ylim(0, 1)
ax.legend(ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.1))
fig.tight_layout(); fig.savefig(FIG / "main_results.png", bbox_inches="tight"); plt.show()

## 3. When does the model lean on the image?

Gated fusion's `image_reliance` = how much each prediction used the image (0 = text only, 1 = image only).
Averaged by true humanitarian class on the test split.

In [ ]:
df, _ = load_predictions(run_dir("fusion_gated") / "predictions_test.csv", TASKS)
names = label_maps(["humanitarian"])["humanitarian"]
hum = df[df.y_humanitarian != -100].assign(cls=lambda d: d.y_humanitarian.map(lambda i: names[i].replace("_", " ")))
order = hum.groupby("cls").image_reliance.median().sort_values().index
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.boxplot([hum[hum.cls == c].image_reliance for c in order], vert=False, widths=0.5, showfliers=False,
           patch_artist=True, boxprops={"facecolor": "#cde2fb", "edgecolor": "#2a78d6"},
           medianprops={"color": "#184f95", "linewidth": 2}, whiskerprops={"color": "#8c8b86"},
           capprops={"color": "#8c8b86"})
ax.set_yticks(range(1, len(order) + 1), order)
ax.set_xlabel("Image reliance (0 = text only, 1 = image only)"); ax.set_xlim(0, 1)
fig.tight_layout(); fig.savefig(FIG / "image_reliance.png", bbox_inches="tight"); plt.show()
hum.groupby("cls").image_reliance.describe()[["count", "mean", "50%"]].round(3)

## 4. Calibration: reliability before and after temperature scaling (humanitarian)

In [ ]:
ev = json.loads((run_dir("fusion_gated") / "evaluation.json").read_text())
_, preds = load_predictions(run_dir("fusion_gated") / "predictions_test.csv", TASKS)
logits, y = preds["humanitarian"]
T = ev["calibration"]["humanitarian"]["temperature"]

def reliability(p, y, bins=10):
    conf, correct = p.max(1), p.argmax(1) == y
    edges = np.linspace(0, 1, bins + 1)
    rows = [(conf[(conf > lo) & (conf <= hi)].mean(), correct[(conf > lo) & (conf <= hi)].mean())
            for lo, hi in zip(edges[:-1], edges[1:]) if ((conf > lo) & (conf <= hi)).sum() >= 5]
    return np.array(rows) if rows else np.zeros((0, 2))

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot([0, 1], [0, 1], ls="--", color="#8c8b86", lw=1, label="perfect")
for label, temp, color in (("before", 1.0, "#eb6834"), (f"after (T={T:.2f})", T, "#2a78d6")):
    r = reliability(softmax(logits, temp), y)
    if len(r):
        ax.plot(r[:, 0], r[:, 1], marker="o", ms=5, lw=2, color=color, label=label)
c_ = ev["calibration"]["humanitarian"]
ax.set_title(f"ECE {c_['test_ece_before']:.3f} → {c_['test_ece_after']:.3f}", fontsize=10)
ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout(); fig.savefig(FIG / "calibration.png", bbox_inches="tight"); plt.show()

## 5. Robustness: missing modalities (gated fusion)

In [ ]:
rob = ev.get("robustness")
if not rob:
    print("No robustness results: run `python -m crisislens.evaluate --run ... --robustness`.")
else:
    conds = [("full", "Text + image", "#6250d6"), ("no_image", "Image hidden", "#2a78d6"), ("no_text", "Text hidden", "#eb6834")]
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for i, (key, label, color) in enumerate(conds):
        vals = [rob[key].get(t, np.nan) for t in TASKS]
        x = np.arange(len(TASKS)) + (i - 1) * 0.26
        ax.bar(x, vals, 0.24, color=color, label=label)
    ax.set_xticks(range(len(TASKS)), [t.title() for t in TASKS]); ax.set_ylim(0, 1); ax.set_ylabel("Test macro-F1")
    ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.15))
    fig.tight_layout(); fig.savefig(FIG / "robustness.png", bbox_inches="tight"); plt.show()

## 6. Where the deployed model gets confused (humanitarian, test)

In [ ]:
m = json.loads((run_dir("fusion_gated") / "metrics.json").read_text())["test"]["humanitarian"]
cm = np.array(m["confusion_matrix"], float); norm = cm / cm.sum(1, keepdims=True).clip(1)
short = {"affected_individuals": "Affected", "infrastructure_and_utility_damage": "Infrastructure",
         "not_humanitarian": "Not humanitarian", "other_relevant_information": "Other info",
         "rescue_volunteering_or_donation_effort": "Rescue/donation"}
labels = [short.get(l, l) for l in m["labels"]]
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(norm, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right"); ax.set_yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{norm[i, j]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if norm[i, j] > 0.55 else "#0b0b0b")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.tight_layout(); fig.savefig(FIG / "confusion_humanitarian.png", bbox_inches="tight"); plt.show()

## 7. Training curves (seed 42)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for model in ["text_bertweet", "image_clip", "fusion_concat", "fusion_gated"]:
    path = run_dir(model) / "metrics.json"
    if path.exists():
        h = pd.DataFrame(json.loads(path.read_text())["history"])
        ax.plot(h.epoch, h.dev_mean_macro_f1, marker="o", ms=5, lw=2, color=COLORS[model], label=PRETTY[model])
ax.set_xlabel("Epoch"); ax.set_ylabel("Dev mean macro-F1"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG / "training_curves.png", bbox_inches="tight"); plt.show()

## 8. Deployment: ONNX export

In [ ]:
report_path = Path("../exported/export_report.json")
if report_path.exists():
    r = json.loads(report_path.read_text())
    display(pd.DataFrame({"size (MB)": r["files_mb"]}).T)
    display(pd.DataFrame({"latency (ms, batch 1)": r["latency_ms"]}).T)
else:
    print("Run export_onnx first.")

## Takeaways (fill in with your numbers)

1. **Does the image help?** Gated fusion vs. BERTweet: humanitarian macro-F1 __ → __ (__ points).
2. **Learned vs. simple fusion:** gated __ vs. late fusion __ → (worth it / not worth it).
3. **When it uses the image:** highest reliance for __ posts, lowest for __ (makes sense because __).
4. **Calibration:** ECE __ → __; the urgent flag catches __% of urgent posts at __% precision.
5. **Robustness:** with the image hidden, humanitarian F1 drops only __ (modality dropout works).
6. **Unseen disaster:** on the California wildfires, never seen in training, macro-F1 __ vs. __ in-distribution.
7. **Deployment:** int8 ONNX is __ MB and __ ms per post on CPU, __x faster than PyTorch.